# PHASE 0.7 — Corrected decision

## The Phase 0.6 verdict was wrong, and the fault is in my notebook

The verdict cell printed *"NO METHOD CONTRIBUTION"*. The numbers do not support that. Here is
the full results table from your run, with the Phase 0.5 numerical gate applied to **every** arm
rather than only to C3:

| Calibrator | params | NLL | ECE | AURC | T_min | Gate |
|---|---|---|---|---|---|---|
| C0 uncalibrated | 0 | 1.1155 | 0.1222 | 0.0950 | 1.000 | — |
| C1 scalar / clean | 1 | 1.9584 | 0.1674 | 0.0748 | 0.488 | **FAIL** — worse than uncalibrated |
| C2 scalar / augmented | 1 | 0.8270 | 0.1333 | 0.1018 | 3.151 | sound |
| C3 MLP / raw *q* | 833 | 1.5428 | 0.1484 | 0.1380 | **0.0100** | **FAIL** — collapsed, worse than uncalibrated |
| **C4 linear / raw *q*** | 9 | **0.7677** | **0.0910** | **0.0884** | 1.153 | sound |
| C5 linear / class one-hot | 7 | 1.0128 | 0.0911 | 0.1034 | 0.396 | sound |
| C6 linear / standardised *q* | 9 | 1.3067 | 0.0956 | 0.1566 | **0.0103** | **FAIL** — collapsed, worse than uncalibrated |

**C6 collapsed onto the epsilon floor.** Its learned temperature contribution is 0.0003 — under
a tenth of the floor — which is exactly the pathology the Phase 0.5 gate was written to catch.
Its NLL is worse than doing no calibration at all. A broken arm carries no information about the
method, and my verdict cell checked `c3_bad` while never testing C6. That is a logic bug: the
notebook concluded "no method contribution" from an arm that had failed numerically.

## What the sound arms actually say

Comparing only the three arms that pass the gate:

**C4 beats C2 on every metric.** NLL 0.7677 vs 0.8270, ECE 0.0910 vs 0.1333, AURC 0.0884 vs
0.1018. Quality-conditioning *does* beat a single scalar temperature.

**C4 beats C5 on NLL by a wide margin** (0.7677 vs 1.0128) **and ties it on ECE** (0.0910 vs
0.0911, a difference of 0.0001).

That second line carries more weight than it appears to, because of what C5 is. C5 conditions the
temperature on the **one-hot predicted class and nothing else** — it holds the class information
*perfectly*, and better than the descriptor does (the model predicts at 0.795, the descriptor at
0.737). So any advantage C4 has over C5 cannot be class information; C5 already has more of it.
C4's NLL advantage must come from variation *within* a class, which is the quality signal.

**This reframes the leakage problem.** The right answer to *"your temperature is conditioning on
class, not quality"* turns out to be a control, not a transformation of the input. C5 answers the
objection directly and cannot break. The class-conditional standardiser was an attempt to remove
the class information from *q*; C5 makes removing it unnecessary.

The ECE tie is a real caveat and must be reported: measured by ECE alone, a per-class temperature
is as good as a quality-conditioned one. The quality signal shows up in likelihood, not in bin
calibration.

## What this notebook does

1. Applies the numerical gate to **every** arm before any conclusion is drawn, and refuses to
   conclude from a failed arm.
2. Runs the bootstraps Phase 0.6 never ran: **C4 vs C2** and **C4 vs C5**. Phase 0.6 only tested
   C6, the broken arm.
3. Diagnoses why C6 collapsed and tests three repairs.
4. Re-decides on the evidence.

## Why C6 probably collapsed

The standardiser's μ and σ were estimated on **clean** training descriptors. At test time it is
applied to images under severity-5 corruption, which sit far outside the clean per-class
distribution — so *z* becomes a large outlier, the linear head is driven into an extreme region,
and the temperature saturates at the floor. Mispredicted classes make it worse: when the
predicted class is wrong, *z* is computed against the wrong reference entirely.

Three repairs are tested: fitting the standardiser on the **augmented** development distribution
so its scale reference spans the operating range, a variance floor plus clipping so no single
dimension can explode, and a **bounded** temperature parameterisation that makes collapse
structurally impossible.

## 0. Self-contained core module

In [ ]:
CORE_V2 = r"""
import math, hashlib
import numpy as np, cv2

def _clip(x): return np.clip(x, 0, 1)
def _mask3(mask, x): return mask[..., None] if x.ndim == 3 else mask

# ------------------------------------------------------------------ optics --
def defocus_blur(x, s):
    r = [1, 2, 3, 5, 7][s-1]
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1)).astype(np.float32)
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def motion_blur(x, s):
    ksz = [5, 9, 13, 19, 25][s-1]
    ang = np.random.uniform(0, 180)
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), ang, 1.0)
    k = cv2.warpAffine(k, M, (ksz, ksz))
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def vignetting(x, s):
    st = [0.15, 0.30, 0.45, 0.62, 0.80][s-1]
    h, w = x.shape[:2]; yy, xx = np.mgrid[0:h, 0:w]
    r = np.sqrt(((xx-w/2)/(w/2))**2 + ((yy-h/2)/(h/2))**2)
    m = np.clip(1 - st*np.clip(r-0.4, 0, None)/0.6, 0, 1).astype(np.float32)
    return _clip(x * _mask3(m, x))

def lens_contamination(x, s):
    n = [3, 7, 13, 22, 34][s-1]; h, w = x.shape[:2]
    blurred = cv2.GaussianBlur(x, (0, 0), sigmaX=max(1.0, min(h, w)/40))
    mask = np.zeros((h, w), np.float32)
    for _ in range(n):
        c = (np.random.randint(0, w), np.random.randint(0, h))
        rad = np.random.randint(max(2, int(0.015*w)), max(4, int(0.07*w)))
        cv2.circle(mask, c, rad, 1.0, -1, lineType=cv2.LINE_AA)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(1.0, min(h, w)/60))
    m = _mask3(mask, x)                       # guard: 2-D input used to broadcast to (H,W,W)
    return _clip((x*(1-m) + blurred*m) * (1 - 0.25*m))

def vibration_jitter(x, s):
    amp = [0.6, 1.4, 2.6, 4.2, 6.5][s-1]; h, w = x.shape[:2]
    dx, dy = np.random.uniform(-amp, amp, 2)
    out = cv2.warpAffine(x, np.float32([[1, 0, dx], [0, 1, dy]]), (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    ksz = int(max(3, 2*round(amp)+1))
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M2 = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), math.degrees(math.atan2(dy, dx)), 1.0)
    k = cv2.warpAffine(k, M2, (ksz, ksz))
    return _clip(cv2.filter2D(out, -1, k/max(k.sum(), 1e-8)))

# ------------------------------------------------------------------ sensor --
def gaussian_noise(x, s):
    return _clip(x + np.random.normal(0, [0.03, 0.06, 0.10, 0.16, 0.24][s-1], x.shape))

def shot_noise(x, s):
    lam = [80, 35, 15, 7, 3][s-1]
    return _clip(np.random.poisson(x*lam)/float(lam))

def scanline_banding(x, s):
    amp = [0.03, 0.06, 0.11, 0.17, 0.25][s-1]; h = x.shape[0]
    period = np.random.uniform(3, 22); phase = np.random.uniform(0, 2*np.pi)
    b = (1 + amp*np.sin(2*np.pi*np.arange(h)/period + phase)).astype(np.float32)
    return _clip(x * (b[:, None, None] if x.ndim == 3 else b[:, None]))

# ------------------------------------------------------------- photometric --
def illumination_gradient(x, s):
    st = [0.10, 0.20, 0.32, 0.46, 0.62][s-1]; h, w = x.shape[:2]
    ang = np.random.uniform(0, 2*np.pi); yy, xx = np.mgrid[0:h, 0:w]
    u = (xx/w-.5)*np.cos(ang) + (yy/h-.5)*np.sin(ang)
    g = (1 + st*u/(np.abs(u).max()+1e-8)).astype(np.float32)
    return _clip(x * _mask3(g, x))

def brightness_drift(x, s):
    # v2: gamma instead of an additive offset. Gamma maps [0,1] -> [0,1] and CANNOT clip.
    # v1 saturated 36.8% of pixels at severity 5 on Magnetic Tile, which destroyed dynamic
    # range rather than shifting brightness and made the top of the ladder meaningless.
    g = [1.12, 1.26, 1.45, 1.72, 2.05][s-1]
    if np.random.rand() < 0.5: g = 1.0/g
    return _clip(np.power(_clip(x), g))

def contrast_loss(x, s):
    g = [0.80, 0.65, 0.50, 0.36, 0.24][s-1]
    m = x.mean(axis=(0, 1), keepdims=True)
    return _clip((x-m)*g + m)

# ---------------------------------------------------------------- pipeline --
def jpeg_compression(x, s):
    q = [70, 50, 32, 18, 9][s-1]
    src = (x[..., ::-1]*255).astype(np.uint8) if x.ndim == 3 else (x*255).astype(np.uint8)
    _, enc = cv2.imencode(".jpg", src, [int(cv2.IMWRITE_JPEG_QUALITY), q])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR if x.ndim == 3 else cv2.IMREAD_GRAYSCALE)
    return (dec[..., ::-1] if x.ndim == 3 else dec).astype(np.float32)/255.

CORRUPTIONS = {
    "defocus_blur": defocus_blur, "motion_blur": motion_blur, "vignetting": vignetting,
    "lens_contamination": lens_contamination, "vibration_jitter": vibration_jitter,
    "gaussian_noise": gaussian_noise, "shot_noise": shot_noise,
    "scanline_banding": scanline_banding, "illumination_gradient": illumination_gradient,
    "brightness_drift": brightness_drift, "contrast_loss": contrast_loss,
    "jpeg": jpeg_compression,
}
TRAIN_FAMILIES = ["defocus_blur", "gaussian_noise", "illumination_gradient",
                  "jpeg", "lens_contamination", "scanline_banding"]
TEST_FAMILIES  = ["motion_blur", "shot_noise", "brightness_drift",
                  "contrast_loss", "vignetting", "vibration_jitter"]
SEVERITIES = [1, 2, 3, 4, 5]
CONDITIONS = [("clean", 0)] + [(f, s) for f in CORRUPTIONS for s in SEVERITIES]


def corruption_seed(image_id, family, severity=None):
    # Depends on (image_id, family) only, NOT severity: nuisance parameters (blur angle,
    # banding period, blob positions, gradient orientation) stay fixed so that severity is
    # the sole varying factor. Seeding on severity too made scanline_banding score 0.20.
    h = hashlib.sha256(f"{image_id}|{family}".encode()).digest()
    return int.from_bytes(h[:4], "little")


def apply_corruption(img_u8, family, severity, image_id=None, seed=None):
    if family == "clean":
        return img_u8
    if seed is None and image_id is not None:
        seed = corruption_seed(image_id, family)
    st = None
    if seed is not None:
        st = np.random.get_state(); np.random.seed(seed % (2**32))
    try:
        out = CORRUPTIONS[family](img_u8.astype(np.float32)/255., severity)
    finally:
        if st is not None: np.random.set_state(st)
    return (np.clip(out, 0, 1)*255).astype(np.uint8)


# ------------------------------------------------------- quality descriptors -
def _gray(im):
    g = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY) if im.ndim == 3 else im
    return g.astype(np.float32)/255.

def _sharp(im): return float(cv2.Laplacian(_gray(im), cv2.CV_32F).var())

def _noise(im):
    g = _gray(im); h, w = g.shape
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    return float(np.abs(cv2.filter2D(g, -1, M)).sum()*math.sqrt(math.pi/2) /
                 (6*max(w-2, 1)*max(h-2, 1)))

def _block(im):
    dh = np.abs(np.diff(_gray(im), axis=1))
    on = dh[:, 7::8].mean() if dh.shape[1] > 8 else 0.0
    return float(on/(dh.mean()+1e-8) - 1.0)

def _hf(im):
    g = _gray(im); f = np.abs(np.fft.fftshift(np.fft.fft2(g))); h, w = g.shape
    cy, cx = h//2, w//2; r = max(4, min(h, w)//8)
    return float(1.0 - f[cy-r:cy+r, cx-r:cx+r].sum()/(f.sum()+1e-8))


def quality_descriptor(img_u8):
    # 8-d ABSOLUTE no-reference descriptor (unchanged from v1).
    g = _gray(img_u8); h, w = g.shape
    if img_u8.ndim == 3:
        rg = img_u8[..., 0].astype(np.float32) - img_u8[..., 1]
        yb = .5*(img_u8[..., 0].astype(np.float32) + img_u8[..., 1]) - img_u8[..., 2]
        colour = float((np.sqrt(rg.std()**2 + yb.std()**2)
                        + .3*np.sqrt(rg.mean()**2 + yb.mean()**2))/255.)
    else:
        colour = 0.0
    cen = g[h//4:3*h//4, w//4:3*w//4]
    per = (g.sum()-cen.sum())/max(g.size-cen.size, 1)
    v = np.array([math.log1p(max(_sharp(img_u8), 0.)*1e3), _hf(img_u8),
                  math.log1p(max(_noise(img_u8), 0.)*1e3), _block(img_u8),
                  float(g.mean()), float(g.std()), colour,
                  float(cen.mean()/(per+1e-8))], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


def quality_descriptor_relative(img_u8):
    # 8-d PERTURBATION-RESPONSE descriptor: how much does a statistic move when a known
    # perturbation is applied? Higher severity signal than the absolute set (0.324 vs 0.236
    # on the synthetic benchmark) but NO reduction in class leakage on its own -- ratios
    # remove the absolute texture level, not the spectral shape. Use with class-conditional
    # standardisation, never alone.
    g8 = (_gray(img_u8)*255).astype(np.uint8); eps = 1e-8
    b = cv2.GaussianBlur(g8, (0, 0), 1.5)
    dn = cv2.resize(cv2.resize(g8, (max(2, g8.shape[1]//2), max(2, g8.shape[0]//2)),
                               interpolation=cv2.INTER_AREA),
                    (g8.shape[1], g8.shape[0]), interpolation=cv2.INTER_LINEAR)
    rn = np.random.default_rng(12345)
    nz = np.clip(g8.astype(np.float32) + rn.normal(0, 12, g8.shape), 0, 255).astype(np.uint8)
    _, e = cv2.imencode(".jpg", g8, [int(cv2.IMWRITE_JPEG_QUALITY), 40])
    jp = cv2.imdecode(e, cv2.IMREAD_GRAYSCALE)
    kh = np.zeros((9, 9), np.float32); kh[4, :] = 1/9.
    hb = cv2.filter2D(g8.astype(np.float32), -1, kh).astype(np.uint8)
    vb = cv2.filter2D(g8.astype(np.float32), -1, kh.T).astype(np.uint8)
    x = g8.astype(np.float32)/255.
    v = np.array([
        math.log((_sharp(g8)+eps)/(_sharp(b)+eps)),
        math.log((_sharp(g8)+eps)/(_sharp(dn)+eps)),
        math.log((_noise(nz)+eps)/(_noise(g8)+eps)),
        _block(jp) - _block(g8),
        float(((x+0.25) > 1.0).mean() + ((x-0.25) < 0.0).mean()),
        abs(math.log((_sharp(hb)+eps)/(_sharp(vb)+eps))),
        math.log((_hf(g8)+eps)/(_hf(b)+eps)),
        math.log((_gray(g8).std()+eps)/(_gray(b).std()+eps)),
    ], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


QUALITY_NAMES = ["sharpness", "hf_energy", "noise", "blockiness",
                 "luminance_mean", "luminance_std", "colourfulness", "vignette_ratio"]
RELATIVE_NAMES = ["blur_headroom", "resolution_headroom", "noise_headroom",
                  "compression_headroom", "clipping_headroom", "blur_anisotropy",
                  "hf_retention", "contrast_retention"]
QUALITY_DIM = 8


class ClassConditionalStandardiser:
    # z = (q - mu_c) / sigma_c, with mu_c and sigma_c estimated on DEVELOPMENT data only.
    #
    # Rationale: degradation is relative. A blurry-looking crazing image and a sharp-looking
    # patches image can have identical absolute sharpness; what makes one degraded is that it
    # is blurrier than crazing images normally are. Removing the class-conditional mean strips
    # the content component and leaves the deviation-from-typical -- which is the degradation.
    #
    # At test time c is the model's own prediction. There is no feedback loop: a per-image
    # scalar temperature cannot change the argmax, so the prediction is fixed before the
    # calibrator runs.
    def __init__(self, n_classes):
        self.n_classes = n_classes; self.mu = None; self.sd = None

    def fit(self, q, y):
        d = q.shape[1]
        self.mu = np.zeros((self.n_classes, d), np.float32)
        self.sd = np.ones((self.n_classes, d), np.float32)
        gm, gs = q.mean(0), q.std(0) + 1e-6
        for c in range(self.n_classes):
            m = (y == c)
            if m.sum() >= 5:                 # fall back to global stats for tiny classes
                self.mu[c] = q[m].mean(0); self.sd[c] = q[m].std(0) + 1e-6
            else:
                self.mu[c] = gm; self.sd[c] = gs
        return self

    def transform(self, q, y_pred):
        y_pred = np.asarray(y_pred).astype(int)
        return ((q - self.mu[y_pred]) / self.sd[y_pred]).astype(np.float32)
"""
print(f"embedded core module: {len(CORE_V2.splitlines())} lines")

embedded core module: 246 lines


## 1. Setup, data, features

Identical pipeline to Phase 0.6 so the numbers are directly comparable: same seed, same subset,
same backbone with its own data config, same corruption conditions.

In [ ]:
#@title Dependencies and core module
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","timm==1.0.11","scikit-learn==1.5.2",
                "opencv-python-headless==4.10.0.84","pandas==2.2.3","kagglehub","tqdm"],check=True)

import os, json, math, time, random, hashlib, warnings
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import timm
warnings.filterwarnings("ignore")

SEED=20260821
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
ROOT=Path("/content/sdic"); OUT=ROOT/"phase0"; OUT.mkdir(parents=True,exist_ok=True)
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

sys.path.insert(0,str(ROOT))
_t=ROOT/"sdic_core_v2.py"; _w=hashlib.sha256(CORE_V2.encode()).hexdigest()
if not _t.exists() or hashlib.sha256(_t.read_text().encode()).hexdigest()!=_w:
    _t.write_text(CORE_V2); print(f"sdic_core_v2.py written, sha256 {_w[:16]}")
else: print(f"sdic_core_v2.py matches audited copy, sha256 {_w[:16]}")
import importlib, sdic_core_v2; importlib.reload(sdic_core_v2)
from sdic_core_v2 import (CORRUPTIONS, TRAIN_FAMILIES, TEST_FAMILIES, SEVERITIES,
                          apply_corruption, quality_descriptor, QUALITY_DIM)
print("device:", DEVICE)

sdic_core_v2.py written, sha256 fd8e3c534562f9d2
device: cuda


In [ ]:
#@title Index NEU and extract features
import kagglehub
from timm.data import resolve_model_data_config
_INTERP={"bilinear":cv2.INTER_LINEAR,"bicubic":cv2.INTER_CUBIC,
         "nearest":cv2.INTER_NEAREST,"area":cv2.INTER_AREA}
NEU_ROOT=Path(kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database"))
neu=[p for p in sorted(NEU_ROOT.rglob("*.jpg"))
     if p.parent.name.lower() not in {"train","validation","images","annotations"}]
labels=np.array([p.parent.name for p in neu])
LUT={n:i for i,n in enumerate(sorted(set(labels)))}
y_all=np.array([LUT[l] for l in labels]); NC=len(LUT)
def rd(p): return cv2.cvtColor(cv2.imread(str(p)),cv2.COLOR_BGR2RGB)
print(f"{len(neu)} images, {NC} classes")

BACKBONE="resnet50.a1_in1k"; N_PER_CLASS=150
set_seed()
per=defaultdict(list)
for p,yy in zip(neu,y_all): per[yy].append(p)
sel=[]
for c,v in per.items():
    pick=np.random.default_rng(SEED+c).choice(len(v),min(N_PER_CLASS,len(v)),replace=False)
    sel += [neu.index(v[t]) for t in pick]
sel=np.array(sorted(sel)); S_paths=[neu[i] for i in sel]; S_y=y_all[sel]

model=timm.create_model(BACKBONE,pretrained=True,num_classes=0).eval().to(DEVICE)
_c=resolve_model_data_config(model)
PP={"mean":np.array(_c["mean"],np.float32),"std":np.array(_c["std"],np.float32),"size":224,
    "interp":_INTERP.get(_c["interpolation"],cv2.INTER_CUBIC),
    "crop_pct":float(_c.get("crop_pct") or 1.0)}
def prep(im):
    sz=PP["size"]; to=int(round(sz/PP["crop_pct"])); h,w=im.shape[:2]; s=to/min(h,w)
    r=cv2.resize(im,(max(1,int(round(w*s))),max(1,int(round(h*s)))),interpolation=PP["interp"])
    hh,ww=r.shape[:2]; t,l=(hh-sz)//2,(ww-sz)//2
    x=(r[t:t+sz,l:l+sz].astype(np.float32)/255.-PP["mean"])/PP["std"]
    return torch.from_numpy(x).permute(2,0,1)

CONDS=[("clean",0)]+[(f,s) for f in TRAIN_FAMILIES for s in (1,3,5)] \
                   +[(f,s) for f in TEST_FAMILIES for s in SEVERITIES]

@torch.no_grad()
def extract(fam,sev,bs=96):
    Fs,Qs=[],[]
    for i in range(0,len(S_paths),bs):
        xs,qs=[],[]
        for p in S_paths[i:i+bs]:
            im=apply_corruption(rd(p),fam,max(sev,1),image_id=p.stem) if fam!="clean" else rd(p)
            qs.append(quality_descriptor(im)); xs.append(prep(im))
        with torch.autocast("cuda",enabled=DEVICE=="cuda"):
            Fs.append(model(torch.stack(xs).to(DEVICE)).float().cpu().numpy())
        Qs.append(np.stack(qs))
    return np.concatenate(Fs),np.concatenate(Qs)

t0=time.time(); CACHE={}
for fam,sev in tqdm(CONDS,desc="extract"): CACHE[(fam,sev)]=extract(fam,sev)
del model; torch.cuda.empty_cache()
print(f"{len(CONDS)} conditions in {(time.time()-t0)/60:.1f} min")

Using Colab cache for faster access to the 'neu-surface-defect-database' dataset.
1800 images, 6 classes


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

extract:   0%|          | 0/49 [00:00<?, ?it/s]

49 conditions in 7.4 min


## 2. Repaired standardisers and a bounded temperature

In [ ]:
class RobustClassStandardiser:
    """z = clip((q - mu_c)/sigma_c, +/- clip_z), with mu and sigma estimated per class.

    Three differences from the version that collapsed:
      * fit_on='augmented' lets the scale reference span the operating range instead of clean
        images only, so a severity-5 image is no longer a 30-sigma outlier by construction;
      * sigma is floored at a fraction of the global sigma, so a dimension that happens to be
        near-constant within one class cannot blow up;
      * z is clipped, so a mispredicted class cannot drive the head into saturation.
    """
    def __init__(self, n_classes, sd_floor=0.25, clip_z=5.0):
        self.n=n_classes; self.sd_floor=sd_floor; self.clip_z=clip_z
    def fit(self,q,y):
        d=q.shape[1]; gm,gs=q.mean(0),q.std(0)+1e-6
        self.mu=np.tile(gm,(self.n,1)).astype(np.float32)
        self.sd=np.tile(gs,(self.n,1)).astype(np.float32)
        for c in range(self.n):
            m=(y==c)
            if m.sum()>=20:
                self.mu[c]=q[m].mean(0)
                self.sd[c]=np.maximum(q[m].std(0), self.sd_floor*gs)+1e-6
        return self
    def transform(self,q,y_pred):
        y_pred=np.asarray(y_pred).astype(int)
        z=(q-self.mu[y_pred])/self.sd[y_pred]
        return np.clip(z,-self.clip_z,self.clip_z).astype(np.float32)


EPS=1e-2
class ScalarT(nn.Module):
    def __init__(self,d=None):
        super().__init__(); self.log_t=nn.Parameter(torch.zeros(()))
    def temperature(self,q): return self.log_t.exp().expand(q.shape[0])+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class LinearT(nn.Module):
    def __init__(self,d,wd=0.0):
        super().__init__()
        self.register_buffer("mu",torch.zeros(d)); self.register_buffer("sd",torch.ones(d))
        self.lin=nn.Linear(d,1); nn.init.zeros_(self.lin.weight)
        nn.init.constant_(self.lin.bias,math.log(math.exp(1.0-EPS)-1.0))
    def fit_norm(self,q): self.mu.copy_(q.mean(0)); self.sd.copy_(q.std(0).clamp_min(1e-6))
    def temperature(self,q): return F.softplus(self.lin((q-self.mu)/self.sd).squeeze(-1))+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class BoundedT(nn.Module):
    """T = T_lo + (T_hi - T_lo) * sigmoid(w.z + b). Collapse is structurally impossible."""
    def __init__(self,d,t_lo=0.5,t_hi=5.0):
        super().__init__(); self.t_lo,self.t_hi=t_lo,t_hi
        self.register_buffer("mu",torch.zeros(d)); self.register_buffer("sd",torch.ones(d))
        self.lin=nn.Linear(d,1); nn.init.zeros_(self.lin.weight)
        p=(1.0-t_lo)/(t_hi-t_lo)
        nn.init.constant_(self.lin.bias,math.log(p/(1-p)))
    def fit_norm(self,q): self.mu.copy_(q.mean(0)); self.sd.copy_(q.std(0).clamp_min(1e-6))
    def temperature(self,q):
        return self.t_lo+(self.t_hi-self.t_lo)*torch.sigmoid(
            self.lin((q-self.mu)/self.sd).squeeze(-1))
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class MLPT(LinearT):
    def __init__(self,d,h=32):
        super().__init__(d); self.lin=None
        self.net=nn.Sequential(nn.Linear(d,h),nn.SiLU(),nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.constant_(self.net[-1].bias,math.log(math.exp(1.0-EPS)-1.0))
    def temperature(self,q): return F.softplus(self.net((q-self.mu)/self.sd).squeeze(-1))+EPS

def fit_cal(cls,L,Q,Y,epochs=400,lr=1e-2,wd=0.0,**kw):
    set_seed()
    L=torch.as_tensor(L,dtype=torch.float32); Q=torch.as_tensor(Q,dtype=torch.float32)
    Y=torch.as_tensor(Y,dtype=torch.long)
    m=cls(Q.shape[1],**kw)
    if hasattr(m,"fit_norm"): m.fit_norm(Q)
    opt=torch.optim.Adam(m.parameters(),lr=lr,weight_decay=wd)
    for _ in range(epochs):
        opt.zero_grad(); F.cross_entropy(m(L,Q),Y).backward(); opt.step()
    return m.eval()

# sanity: every head must be an exact no-op at initialisation
for cls,kw in [(LinearT,{}),(BoundedT,{}),(MLPT,{})]:
    m=cls(8,**kw); m.fit_norm(torch.randn(200,8))
    T=m.temperature(torch.randn(200,8)).detach()
    assert abs(T.mean().item()-1.0)<1e-4 and T.std().item()<1e-5, cls.__name__
print("all temperature heads initialise to T = 1.0 exactly")

all temperature heads initialise to T = 1.0 exactly


## 3. Probe, calibration sets, and all arms

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import balanced_accuracy_score

skf=StratifiedKFold(5,shuffle=True,random_state=SEED)
tr_all,te=next(skf.split(np.zeros(len(S_y)),S_y))
tr,va=train_test_split(tr_all,test_size=0.25,random_state=SEED,stratify=S_y[tr_all])

Fc,Qc=CACHE[("clean",0)]
probe=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,class_weight="balanced"))
probe.fit(Fc[tr],S_y[tr])
def logits_of(Fx):
    d=probe.decision_function(Fx); return d if d.ndim>1 else np.stack([-d,d],1)

CAL_CONDS=[("clean",0)]+[(f,s) for f in TRAIN_FAMILIES for s in (1,3,5)]
TEST_CONDS=[(f,s) for f in TEST_FAMILIES for s in SEVERITIES]

def gather(idx,conds):
    L,Q,Y=[],[],[]
    for fam,sev in conds:
        Fx,Qx=CACHE[(fam,sev)]
        L.append(logits_of(Fx[idx])); Q.append(Qx[idx]); Y.append(S_y[idx])
    return np.concatenate(L),np.concatenate(Q),np.concatenate(Y)

Ltr_a,Qtr_a,Ytr_a = gather(tr,CAL_CONDS)      # augmented DEV, for fitting standardisers
Lva_a,Qva_a,Yva_a = gather(va,CAL_CONDS)      # augmented VAL, for fitting calibrators
Lva_c,Qva_c,Yva_c = gather(va,[("clean",0)])
Lte,Qte,Yte       = gather(te,TEST_CONDS)
print(f"dev(aug) {len(Ytr_a)} | cal(aug) {len(Yva_a)} | cal(clean) {len(Yva_c)} | test {len(Yte)}")

def onehot(L):
    h=np.zeros((len(L),NC),np.float32); h[np.arange(len(L)),L.argmax(1)]=1.0; return h

# standardiser variants
STD_clean = RobustClassStandardiser(NC,sd_floor=0.0,clip_z=1e9).fit(Qc[tr],S_y[tr])   # as broken
STD_aug   = RobustClassStandardiser(NC,sd_floor=0.0,clip_z=1e9).fit(Qtr_a,Ytr_a)
STD_rob   = RobustClassStandardiser(NC,sd_floor=0.25,clip_z=5.0).fit(Qtr_a,Ytr_a)

def Z(std,Q,L): return std.transform(Q,L.argmax(1))

ARMS={}
ARMS["C2 scalar / augmented"]      = (fit_cal(ScalarT,Lva_a,Qva_a,Yva_a), Qte)
ARMS["C3 MLP / raw q"]             = (fit_cal(MLPT,Lva_a,Qva_a,Yva_a), Qte)
ARMS["C4 linear / raw q"]          = (fit_cal(LinearT,Lva_a,Qva_a,Yva_a), Qte)
ARMS["C5 linear / class one-hot"]  = (fit_cal(LinearT,Lva_a,onehot(Lva_a),Yva_a), onehot(Lte))
ARMS["C6 linear / std (clean-fit)"]= (fit_cal(LinearT,Lva_a,Z(STD_clean,Qva_a,Lva_a),Yva_a),
                                      Z(STD_clean,Qte,Lte))
ARMS["C6b linear / std (aug-fit)"] = (fit_cal(LinearT,Lva_a,Z(STD_aug,Qva_a,Lva_a),Yva_a),
                                      Z(STD_aug,Qte,Lte))
ARMS["C6c linear / std (robust)"]  = (fit_cal(LinearT,Lva_a,Z(STD_rob,Qva_a,Lva_a),Yva_a),
                                      Z(STD_rob,Qte,Lte))
ARMS["C7 bounded / std (robust)"]  = (fit_cal(BoundedT,Lva_a,Z(STD_rob,Qva_a,Lva_a),Yva_a),
                                      Z(STD_rob,Qte,Lte))
ARMS["C7q bounded / raw q"]        = (fit_cal(BoundedT,Lva_a,Qva_a,Yva_a), Qte)
ARMS["C1 scalar / clean"]          = (fit_cal(ScalarT,Lva_c,Qva_c,Yva_c), Qte)
print(f"{len(ARMS)} arms fitted")

print("\nz-scale diagnosis (why C6 collapsed):")
for nm,std in [("clean-fit",STD_clean),("aug-fit",STD_aug),("robust",STD_rob)]:
    z=Z(std,Qte,Lte)
    print(f"  {nm:10s} |z| max {np.abs(z).max():9.1f}   p99 {np.quantile(np.abs(z),0.99):7.2f}   "
          f"fraction |z|>5: {float((np.abs(z)>5).mean()):.3f}")

dev(aug) 10260 | cal(aug) 3420 | cal(clean) 180 | test 5400
10 arms fitted

z-scale diagnosis (why C6 collapsed):
  clean-fit  |z| max  651943.0   p99 325521.53   fraction |z|>5: 0.167
  aug-fit    |z| max      47.1   p99    8.41   fraction |z|>5: 0.019
  robust     |z| max       5.0   p99    5.00   fraction |z|>5: 0.000


## 4. Evaluation with the gate applied to every arm

In [ ]:
def softmax(z):
    z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def ece_em(p,y,nb=15):
    conf,pred=p.max(1),p.argmax(1); corr=(pred==y).astype(float); o=np.argsort(conf)
    return float(sum(len(c)/len(y)*abs(corr[c].mean()-conf[c].mean())
                     for c in np.array_split(o,nb) if len(c)))
def nll_pi(p,y): return -np.log(np.clip(p[np.arange(len(y)),y],1e-12,None))
def aurc(p,y):
    conf,pred=p.max(1),p.argmax(1); e=(pred!=y).astype(float)[np.argsort(-conf)]
    n=len(e); return float(np.trapezoid(np.cumsum(e)/np.arange(1,n+1),np.arange(1,n+1)/n))

p0=softmax(Lte); PIM={"C0 uncalibrated":nll_pi(p0,Yte)}
rows={"C0 uncalibrated":dict(n_params=0,nll=float(PIM["C0 uncalibrated"].mean()),
      ece=ece_em(p0,Yte),aurc=aurc(p0,Yte),acc=float((p0.argmax(1)==Yte).mean()),
      T_min=1.0,T_max=1.0)}
for nm,(m,X) in ARMS.items():
    Xt=torch.as_tensor(X,dtype=torch.float32)
    with torch.no_grad():
        out=m(torch.as_tensor(Lte,dtype=torch.float32),Xt).numpy()
        T=m.temperature(Xt).numpy()
    p=softmax(out); PIM[nm]=nll_pi(p,Yte)
    rows[nm]=dict(n_params=sum(q.numel() for q in m.parameters()),nll=float(PIM[nm].mean()),
                  ece=ece_em(p,Yte),aurc=aurc(p,Yte),acc=float((p.argmax(1)==Yte).mean()),
                  T_min=float(T.min()),T_max=float(T.max()))
R=pd.DataFrame(rows).T

UNCAL=R.loc["C0 uncalibrated","nll"]
def gate(r,name):
    if name=="C0 uncalibrated": return "reference"
    p=[]
    if isinstance(r.T_min,float) and (r.T_min-EPS)<0.1*EPS and r.T_min<1.0:
        p.append("collapsed")
    if r.T_max>50: p.append("exploded")
    if r.nll>UNCAL: p.append("worse than uncalibrated")
    return "; ".join(p) if p else "sound"
R["gate"]=[gate(r,n) for n,r in R.iterrows()]
R["sound"]=R.gate.isin(["sound","reference"])
R.index.name="calibrator"
display(R.round(4))
R.to_csv(OUT/"phase0_7_arms.csv")
print(f"\nsound arms: {list(R[R.sound & (R.index!='C0 uncalibrated')].index)}")

,n_params,nll,ece,aurc,acc,T_min,T_max,gate,sound
calibrator,,,,,,,,,
C0 uncalibrated,0.0,1.1155,0.1222,0.0950,0.7946,1.0000,1.0000,reference,True
C2 scalar / augmented,1.0,0.8270,0.1333,0.1018,0.7946,3.1510,3.1510,sound,True
C3 MLP / raw q,833.0,1.5428,0.1484,0.1380,0.7946,0.0100,18.4151,collapsed; worse than uncalibrated,False
C4 linear / raw q,9.0,0.7677,0.0910,0.0884,0.7946,1.1525,6.9216,sound,True
C5 linear / class one-hot,7.0,1.0128,0.0911,0.1034,0.7946,0.3964,3.7897,sound,True
C6 linear / std (clean-fit),9.0,1.3067,0.0956,0.1566,0.7946,0.0103,4.5725,collapsed; worse than uncalibrated,False
C6b linear / std (aug-fit),9.0,1.3202,0.0910,0.1460,0.7946,0.0101,4.4829,collapsed; worse than uncalibrated,False
C6c linear / std (robust),9.0,0.8934,0.0712,0.1088,0.7946,0.2172,4.3461,sound,True
C7 bounded / std (robust),9.0,0.8629,0.0862,0.1076,0.7946,0.6175,4.3299,sound,True



sound arms: ['C2 scalar / augmented', 'C4 linear / raw q', 'C5 linear / class one-hot', 'C6c linear / std (robust)', 'C7 bounded / std (robust)', 'C7q bounded / raw q']


In [ ]:
#@title The bootstraps Phase 0.6 never ran
def paired_boot(a,b,n_boot=10000,seed=SEED):
    d=a-b; rng=np.random.default_rng(seed); n=len(d)
    bs=np.array([d[rng.integers(0,n,n)].mean() for _ in range(n_boot)])
    return float(d.mean()),float(np.quantile(bs,0.025)),float(np.quantile(bs,0.975))

def ece_boot(na,nb,n_boot=2000,seed=SEED):
    rng=np.random.default_rng(seed); n=len(Yte); out=[]
    def pr(nm):
        if nm=="C0 uncalibrated": return softmax(Lte)
        m,X=ARMS[nm]
        with torch.no_grad():
            return softmax(m(torch.as_tensor(Lte,dtype=torch.float32),
                             torch.as_tensor(X,dtype=torch.float32)).numpy())
    pa,pb=pr(na),pr(nb)
    for _ in range(n_boot):
        s=rng.integers(0,n,n); out.append(ece_em(pa[s],Yte[s])-ece_em(pb[s],Yte[s]))
    o=np.array(out); return float(o.mean()),float(np.quantile(o,0.025)),float(np.quantile(o,0.975))

best_q=max([n for n in R.index if R.loc[n,"sound"] and n.startswith(("C4","C6","C7"))],
           key=lambda n:-R.loc[n,"nll"]) if any(
           R.loc[n,"sound"] and n.startswith(("C4","C6","C7")) for n in R.index) else None
best_q=min([n for n in R.index if R.loc[n,"sound"] and n.startswith(("C4","C6","C7"))],
           key=lambda n:R.loc[n,"nll"])
print(f"best sound quality-conditioned arm: {best_q}\n")

tests=[]
for B,q in [("C2 scalar / augmented","Does quality-conditioning beat one scalar?"),
            ("C5 linear / class one-hot","Is the gain quality, or just content?")]:
    d,lo,hi=paired_boot(PIM[best_q],PIM[B]); e,el,eh=ece_boot(best_q,B)
    tests.append({"A":best_q,"B":B,"question":q,"delta_nll":round(d,4),
                  "nll_ci":f"[{lo:.4f}, {hi:.4f}]","A_wins_nll":bool(hi<0),
                  "delta_ece":round(e,4),"ece_ci":f"[{el:.4f}, {eh:.4f}]",
                  "A_wins_ece":bool(eh<0)})
T=pd.DataFrame(tests); display(T)
T.to_csv(OUT/"phase0_7_decision.csv",index=False)
print("Negative delta favours A. A CI entirely below zero is a win.")

best sound quality-conditioned arm: C4 linear / raw q



,A,B,question,delta_nll,nll_ci,A_wins_nll,delta_ece,ece_ci,A_wins_ece
0,C4 linear / raw q,C2 scalar / augmented,Does quality-conditioning beat one scalar?,-0.0593,"[-0.0654, -0.0535]",True,-0.0419,"[-0.0444, -0.0388]",True
1,C4 linear / raw q,C5 linear / class one-hot,"Is the gain quality, or just content?",-0.2451,"[-0.2976, -0.1948]",True,0.0000,"[-0.0085, 0.0086]",False


Negative delta favours A. A CI entirely below zero is a win.


In [ ]:
#@title Corrected verdict
A=T.iloc[0]["A"]
beats_c2_nll=bool(T.iloc[0]["A_wins_nll"]); beats_c2_ece=bool(T.iloc[0]["A_wins_ece"])
beats_c5_nll=bool(T.iloc[1]["A_wins_nll"]); beats_c5_ece=bool(T.iloc[1]["A_wins_ece"])
broken=[n for n in R.index if not R.loc[n,"sound"]]

print("="*88)
print(f"best sound quality-conditioned arm : {A}")
print(f"beats C2 (scalar)   NLL {beats_c2_nll} | ECE {beats_c2_ece}")
print(f"beats C5 (content)  NLL {beats_c5_nll} | ECE {beats_c5_ece}")
print(f"arms failing the numerical gate    : {broken}")
print("="*88)

if beats_c2_nll and beats_c5_nll:
    v="METHOD CONTRIBUTION SUPPORTED"
    g=(f"{A} beats both a single scalar and the content-only control on held-out corruption "
       f"families. C5 is the load-bearing control: it holds the predicted class exactly, and "
       f"better than the descriptor does, so the advantage over it cannot be class information. "
       f"Report C5 in the main table, not the appendix. If the ECE comparison against C5 is a "
       f"tie, say so plainly -- the quality signal lives in likelihood, not in bin calibration.")
elif beats_c2_nll and not beats_c5_nll:
    v="PARTIAL — beats a scalar, not the content control"
    g=("The temperature is exploiting the predicted class rather than image quality. Rename the "
       "method to class-conditioned calibration and argue it on those terms, or drop it and "
       "make this a benchmark paper.")
else:
    v="NO METHOD CONTRIBUTION — reframe as a benchmark paper"
    g=("Quality-conditioning does not beat a one-parameter scalar. That is a reportable finding: "
       "under acquisition shift, scalar temperature scaling on augmented validation data is "
       "sufficient. Reframe around SDI-C, the split protocol, the leakage precondition and the "
       "corrected severity ladders.")

if broken:
    g += (f"\n\nNOTE: {len(broken)} arm(s) failed the numerical gate and were excluded from the "
          f"decision: {broken}. A failed arm is evidence about the optimisation, not about the "
          f"method, and must never drive a verdict -- that was the bug in Phase 0.6.")
print(f"\nVERDICT: {v}\n"); print(g)
json.dump({"verdict":v,"guidance":g,"best_arm":A,"broken_arms":broken,
           "results":R.round(5).astype(str).to_dict(),"tests":T.to_dict("records")},
          open(OUT/"phase0_7_verdict.json","w"),indent=2)

best sound quality-conditioned arm : C4 linear / raw q
beats C2 (scalar)   NLL True | ECE True
beats C5 (content)  NLL True | ECE False
arms failing the numerical gate    : ['C3 MLP / raw q', 'C6 linear / std (clean-fit)', 'C6b linear / std (aug-fit)', 'C1 scalar / clean']

VERDICT: METHOD CONTRIBUTION SUPPORTED

C4 linear / raw q beats both a single scalar and the content-only control on held-out corruption families. C5 is the load-bearing control: it holds the predicted class exactly, and better than the descriptor does, so the advantage over it cannot be class information. Report C5 in the main table, not the appendix. If the ECE comparison against C5 is a tie, say so plainly -- the quality signal lives in likelihood, not in bin calibration.

NOTE: 4 arm(s) failed the numerical gate and were excluded from the decision: ['C3 MLP / raw q', 'C6 linear / std (clean-fit)', 'C6b linear / std (aug-fit)', 'C1 scalar / clean']. A failed arm is evidence about the optimisation, not about the m

---

## Reading this

The decision now rests on the best arm that is **numerically sound**, and any arm that fails the
gate is reported as excluded rather than treated as evidence. That is the single change that
flips the Phase 0.6 conclusion, and it was a bug in my code rather than a finding about your
method.

Whatever the outcome, three things are now true and worth putting in the paper.

**Clean-fit calibration fails under shift, decisively.** C1 fitted on clean validation data
produced T = 0.488 — below 1, *sharpening* a model that was already overconfident — and reached
NLL 1.958 against 1.116 for no calibration at all. Making your model worse than doing nothing is
a strong opening for the introduction.

**The MLP head is dead.** 833 parameters, temperature pinned to the floor, NLL worse than
uncalibrated. Report it in the ablation as evidence against capacity, not as a component.

**C5 is the control that matters.** If quality-conditioning survives it, the leakage objection is
answered directly and the class-conditional standardiser becomes optional machinery rather than a
necessity. If it does not survive it, you have learned that early and cheaply.

## Still blocked, independent of this result

KSDD2 from the official ViCoS release. Magnetic Tile folds rebuilt without the discarded pHash
grouping. The F1–F6 fixes applied to the main notebook. None of these are affected by the
verdict, and none can be skipped.